## Checkpoint 2 
### Buckman, Garcia, McGuire, Lanaghen

## Criteria
- Project Descripiton
- Data Description
- Ethical Data Concerns
- Methods
- Preliminary Results
- Peer Feedback
    - They pointed out that we ignored major contributors of inflow and outflow such as tributaries and evaporation.
        - This would invalidate linear regressions because there are more than two variables to plot against one another.
- List of Completed Milestones
    - New data aquired, aggregated, and cleaned
    - Narrowed in on analysis plan
    - New dataframe is being created
- List of Methods Milestones

**General Update**

Moving through peer evalutations, we received feedback noting that our original plan failed to account for water flowing in from major tributaries and increased human consumption. We originally wanted to only work with precipitation and snowpack data, but because snowpack can tie into tributary flow data, we decided to pivot and work with tributary flow and precipitation data. We will stick with our origial data source for precipitation data, but we will use the USGS information from the Salt Lake Hydromapper Current Conditions page (https://webapps.usgs.gov/gsl/data.html). This helped us hoene in on what analysis we will do later on, and slightly altered the questions we wanted to answer. 

### New Questions:

1. Can we model future lake outcomes using k-NN sample bootstrapping? 
    - This utilizes historical data in a completely different fashion than what we originally thought we were going to pursue before. 

### 1. Decide data sources

USGS GSL Hydromapper (NEW)
- Started scraping data from the site that has major tributary inflow and outflow data. 
- Our inflow tributaries include three major tributaries, the Bear River (57% of inflow), Weber River (20% of inflow), and the Jordan River (16% of inflow). 
    - There are small caveats associated with the Jordan River as it is diverted through a surplus canal in Salt Lake City, but we are working with the data we can scrape from the USGS site. 

USGS National Water Information System (NWIS) REST API
- Lake water surface elevation at the Saltair Boat Harbor monitoring station (1950-2024)
- Documentation available at https://waterservices.usgs.gov/
- No account required to access the API. Reviewed the terms at https://www.usgs.gov/information-policies-and-instructions/acknowledging-or-crediting-usgs to ensure proper use and crediting of data used
Natural Resources Conservation Service (NRCS) SNOTEL network


Historical snow water equivalent data (relevant snowpack -> runoff estimate) (1980-2026)
- Downloadable CSV available at: https://wcc.sc.egov.usda.gov/reportGenerator/
- Reviewed documentation and terms to ensure proper use


NOAA Climate Data Online (CDO) REST API
- Historical climate and precipitation data (1950-2024)
- Will primarily use the Salt Lake City Airport monitoring station
- Reviewed documentation and terms to ensure proper use https://www.ncei.noaa.gov/cdo-web/webservices/v2. A free account is required to obtain an API token, which has been obtained. The API has a restriction of one year of data per call, so years will be looped through with a rate-limiting pause between requests to remain in compliance.

Great Salt Lake Elevation Area Volume Tables (NEW)
- Contains elevation-area-volume relationships for the Great Salt Lake
- Downloadable CSV avilable at: https://www.sciencebase.gov/catalog/item/6467b42fd34ec11ae4a8afb1, labeled as Great_Salt_Lake_2023_ElevAreaVolume_total.csv
- Root, J.C., 2023, Half-meter topobathymetric elevation model and elevation-area-volume tables for Great Salt Lake, Utah, 2002-2016: U.S. Geological Survey data release, https://doi.org/10.5066/P9DGG75W.
___
### 2. Acquire/scrape data
Data was obtained from the USGS API through an API request and was combined into two CSVs, one for Inflow sources and one for outflow streams going back to 1950. Lake level and precipitation data were collected through separate API requests via the NWIS REST API and NOAA CDO REST API respectively and combined into separate CSV files. The data for both precipitation and lake level stretched back to 1950, simlar to ouflow data. Historical snowpack data was directly downloaded to CSV format from a USDA website going back to 1980. We were able to obtain data from each of our sources in a monthly format which allowed us to look into consolidating into a single data frame. After reviewing our final data sets, we learned that lake outflow data refers to water flow through small, shallow areas of the lake such as Farmington Bay and Willard Bay. As a result we decided not to include this file in our final data frame. Instead we found a new data source for evaporation and decided to use it as a replacement for outflow.

EVAPORATION DATA<br>
We used Bigalke et al.'s (2025) mass-balance equation as a model to calculate and create our own evaporation dataset. We already had key variables such as, inflow, precipitation, and lake elevation data. In order to derive evaporation data, we needed to find the lake volume. We used the USGS Science Data Catalog to find the Great Salt Lake Elevation-Area-Volume tables, which contained elevation-area-volume relationships for the Great Salt Lake. The elevation-area-volume totals were calculated at elevation increments of 0.01 feet between 4,170–4,215 ft (Root, 2023). This data covers the historical range of lake elevations in our dataset.

We started with Bigalke et al.'s (2025) mass-balance equation for calculating lake volume changes:

    V_t = V_(t-1) + P_t × A_(t-1) + R_t − E_t × A_(t-1)
    
Then, we rearranged the equation by solving for E_t, since we opted not to use ERA5 reanalysis data as Bigalke et al. (2025) did. Instead, we used our available data to simplify the mass-balance equation to calculate month-to-month lake volume changes based on inflow, precipitation, and lake elevation. Precipitation was converted from inches to acre-feet by dividing by 12 and multiplying by the prior month's surface area in acres:

    precip_acft = (monthly_precip_inches / 12) × prior_month_surface_area

Since this conversion already accounts for the lake's prior month surface area, A_(t-1) could be eliminated from the equation, resulting in the following simplified equation:

    E_t = V_(t-1) + R_t + P_t − V_t

Where:<br>
* E_t = that month's evaporation (acre-feet)<br>
* V_(t-1) = prior month's lake volume (acre-feet)<br>
* R_t = that month's total inflow (acre-feet)<br>
* P_t = that month's precipitation on the lake surface (acre-feet)<br>
* V_t = current month's lake volume (acre-feet)

### 3. Clean
Thankfully much of the data obtained was already fairly clean, however we took the following steps to ensure compatible structure for eventually combining everything into a single dataframe:
- Created month / year tracking columns for each constituent dataframe
- Renamed columns to be more descriptive or to have similar names across data frames
- Unified column data types (string dates -> DateTimes, numeric conversions, units, etc.)

Discuss what missing values mean and how to accommodate those
---
### 4. Combine all data into one dataframe
Ultimately, we were able to combine the snowpack, lake level, inflow, precipitation, and evaporation data all into a single continuous dataframe which was exported as a new CSV file. Each major category became a column in the final dataframe all binned by month and year. The snowpack column was the only column containing NaN values which occur prior to 1980 (before readings).
___
### 5. Decide on analysis methods

**New Analysis Methods:**

***k-NN Bootstrap Inspired by Lall and Sharma:***

We happened to come across this paper demonostrating a different way of predicting lake level outcomes using historical data. Insead of using k-NN in its predicted fashion of classification based on proximity, it will be used to predict a future outcome based on the feature reads we are given in that particular month or year. We will start with the reads we have right now, then the feature values will be compared to years past. This is where k-NN comes into play. We will use k-NN to compare where we are now to the data in years past to find year N, a year in which the feature reads are closest to where we are now. From there we will observe lake reads in year N+1 which we will detgermine as the outcome of th features combined in year N. 

For this k-NN analysis, we want to use precipitation data, inflow and outflow streams, and general human consumption metrics to create a model that takes into account all of the major players in lake level affectors. But because of the granularity of the model, we will have to take into account model penalties for adding more variables. 

We are choosing this method instead of linear regression for a number of reasons. Due to the large number of variables that contribute to overall lake level. Lake level has not been consistently decreasing at a constant level, so using a simple linear regression would be counterintuitive based on simply visually guaging the plot of lake level over time. 





https://webapps.usgs.gov/gsl/data.html

## Data Acquisition ##

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import requests
import json
import time

In [ ]:
## Perform API request to USGS for lake level data and save the response as a json file
# USGS address for lake level data
usgs_url = "https://waterservices.usgs.gov/nwis/dv/"

# Parameters for API request - we want the lake level data for the Great Salt Lake at Saltair Boat Harbor, UT (USGS-10010000) from 1950 to 2024
usgs_params = {
    "format": "json",    # we want it returned in json
    "sites": "10010000",    # Great Salt Lake at Saltair Boat Harbor, UT - USGS-10010000
    "parameterCd": "62614",   # code for lake water surface elevation
    "startDT": "1950-01-01",   # start and end dates
    "endDT": "2024-12-31"
}

# Getting data from the api address with the params we set above
usgs_response = requests.get(usgs_url, params=usgs_params)
#converting to json
usgs_data = usgs_response.json()

# Save the full json to a file for viewing/searching
with open("usgs_response.json", "w") as f:
    json.dump(usgs_data, f, indent=2)

In [ ]:
## Process and Access the raw lake level data from the json response

# Entering the layers where the data lives, grabbing the first value in each list
usgs_time_series = usgs_data["value"]["timeSeries"][0]["values"][0]["value"]

# Converting to pandas dataframe
usgs_df = pd.DataFrame(usgs_time_series)

# Data exploration
print(usgs_df.head(20))
print(usgs_df.shape)

In [ ]:
# Cleaning up the dataframe
usgs_df = usgs_df.rename(columns={"dateTime": "date", "value": "elevation_ft"})
usgs_df = usgs_df.drop(columns=["qualifiers"])

# Converting the date columns to datetime datatype
usgs_df["date"] = pd.to_datetime(usgs_df["date"])

# Converting the string elevation data to numeric
usgs_df["elevation_ft"] = pd.to_numeric(usgs_df["elevation_ft"], errors="coerce")

# Aggregating to monthly measurements
# Setting the date as the index, grouping by month ("MS" = month start), taking the average of each month, reset the index
usgs_monthly = usgs_df.set_index("date").resample("MS")["elevation_ft"].mean().reset_index()

# Adding some separate date columns for potential later use
usgs_monthly["year"] = usgs_monthly["date"].dt.year
usgs_monthly["month"] = usgs_monthly["date"].dt.month

# Save the cleaned and aggregated data to a new csv file
usgs_monthly.to_csv('lake_level_monthly.csv', index=False)

# Print info about the cleaned lake level data
print(usgs_monthly.info())

# Print a preview of the cleaned lake level data
usgs_monthly

In [ ]:
# Plotting lake elevation over time to see the overall trend and variability in the data

plt.figure(figsize=(14, 5))
plt.plot(usgs_monthly["date"], usgs_monthly["elevation_ft"], linewidth=0.8)
plt.axhline(y=4198, color="green", linestyle="--", label="Healthy level (4,198 ft)")
plt.title("Great Salt Lake Elevation 1950–2024")
plt.ylabel("Elevation (ft above sea level)")
plt.xlabel("Date")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
## Performing API request to NOAA for precipitation data and saving the response as a json file

url = "https://www.ncei.noaa.gov/access/services/data/v1" # Newer version of API that does not require access token (different from proposal)

# NOAA only allows one year per data request, so we loop through each year and gather the data into a single list, which will be converted to a dataframe at the end
all_data = []

for year in range(1950, 2025):
    # Setting up the API request parameters
    params = {
        "dataset": "daily-summaries",
        "stations": "USW00024127",  # SLC airport station code
        "dataTypes": "PRCP", # precipitation data
        "startDate": f"{year}-01-01",   # inserting the current year for each loops date parameters
        "endDate": f"{year}-12-31",
        "format": "json",
        "units": "standard"
    }
    
    # Sending the request
    response = requests.get(url, params=params)
    
    # Checking if the request succeeded (code = 200) and if so converting the response to json data
    if response.status_code == 200:
        data = response.json()
        # Adding data from successful responses, flagging any that didn't work
        if data:
            all_data.extend(data)
            print(f"{year}: got {len(data)} records")
        else:
            print(f"{year}: no results")
    else:
        print(f"{year}: error {response.status_code} - {response.text[:100]}")

    # Adding a delay to abide by API rules
    time.sleep(0.2)

# Converting the list of precipitation data to a dataframe for exploration and cleaning
precip_df = pd.DataFrame(all_data)
print(precip_df.head())
print(precip_df.columns)

In [ ]:
# Cleaning up dataset, similar to NSGS cleaning steps
precip_df = precip_df.rename(columns={"DATE": "date", "PRCP": "precip_inches"})
precip_df["date"] = pd.to_datetime(precip_df["date"])
precip_df["precip_inches"] = pd.to_numeric(precip_df["precip_inches"], errors="coerce")

# Resampling to montly precipitation TOTAL (sum)
# nNote this is different than the lake level aggregation, which was the average measurement (mean)
precip_monthly = precip_df.set_index("date")["precip_inches"].resample("MS").sum().reset_index()
precip_monthly.columns = ["date", "monthly_precip_inches"]

# Add year and month columns for potential later use
precip_monthly["year"] = precip_monthly["date"].dt.year
precip_monthly["month"] = precip_monthly["date"].dt.month

# Creating CSV
precip_monthly.to_csv("precipitation_monthly.csv", index=False)

# Printing info about the cleaned and aggregated precipitation data
print(precip_monthly.info())

# Printing preview of cleaned and aggregated precipitation data
precip_monthly

In [ ]:
## Covert the snowpack data from csv to dataframe and perform cleaning steps similar to the other datasets

# Creating snowpack dataframe to do some cleaning
snowpack_df = pd.read_csv('snowpack.csv', comment='#', skip_blank_lines=True)

# Data cleaning (renaming columns, converting datatypes, adding additional date cols)
snowpack_df = snowpack_df.rename(columns={"Snowbird (766) Snow Water Equivalent (in) Start of Month Values": "snow_water_equiv_in"})
snowpack_df["Date"] = pd.to_datetime(snowpack_df["Date"], format="%b %Y")
snowpack_df["snow_water_equiv_in"] = pd.to_numeric(snowpack_df["snow_water_equiv_in"], errors="coerce")
snowpack_df["year"] = snowpack_df["Date"].dt.year
snowpack_df["month"] = snowpack_df["Date"].dt.month

# Create clean csv file for snowpack
snowpack_df.to_csv('snowpack_monthly.csv')

# Printing info about the cleaned snowpack data
print(snowpack_df.info())

# Printing preview of cleaned snowpack data
snowpack_df

In [ ]:
## Compile snowpack, lake level, and precipitation data sets into a single data frame, then save to a new csv file

# Load each monthly dataframe
precipitation = pd.read_csv("precipitation_monthly.csv")
lake_level = pd.read_csv("lake_level_monthly.csv")
snowpack = pd.read_csv("snowpack_monthly.csv")

# Printed parts of the data frames to see what values overlap and where to join them
# print(snowpack.head(10), "\n")
# print(precipitation.head(10), "\n")
# print(lake_level.head(10), "\n")
# print(lake_level["year"].unique(), "\n")
# print(snowpack["year"].unique(), "\n")

# Renamed the Date column in snowpack to date for consistency
snowpack = snowpack.rename(columns={"Date": "date"})

# Removed 2025 and 2026 data in snowpack to be consistent with other dfs
snowpack = snowpack[snowpack["year"] < 2025]

# Compile the dataframe, joining precipitation with lake_level and snowpack on date, year, and month
# This join strategy assigns NaN where anything doesn't exist (i.e. before 1989 in snowpack data)
compiled_df = precipitation.merge(lake_level, on=["date", "year", "month"], how="outer") \
               .merge(snowpack, on=["date", "year", "month"], how="outer")

# Extract and order the final columns needed in the compiled dataframe
compiled_df = compiled_df[["date", "year", "month", "elevation_ft", "monthly_precip_inches", "snow_water_equiv_in"]]

# Save the final compiled dataframe as a new csv file
compiled_df.to_csv("compiled_lake_data.csv")

# Display the first ten lines of the compiled dataframe
compiled_df.tail(10)

# Note: In the snow_water_equiv_in column, the values are NaN before 1989 since there was no data before
# that time.

### Aquisition and Processing for Inflow/Outflow Data

The largest hurdle with this data was figuring out how to transform the data to be able to use it when it's aggregated monthly. We decided to use acre sq ft per month as our unit when transforming the data. It was also concluded after pulling the data that the outflow bays/sites were not significant enough of a contribution, so in the end we will not be using outflow sites in our analysis.

In [ ]:
import requests
import pandas as pd
from io import StringIO
import os

OUTPUT_DIR = "/Users/kimlanaghen/Downloads"
BASE_URL   = "https://waterservices.usgs.gov/nwis/dv/"


# Inflows  = rivers flowing INTO the lake
# Outflows = diversions / canals taking water AWAY before it reaches the lake

INFLOW_SITES = {
    "Bear River (primary, near Corinne)":  "10126000",
    "Bear River (upstream gap-fill)":      "10118000",
    "Weber River (near Plain City)":       "10141000",
    "Jordan River (Surplus Canal)":        "10170500",
    "Jordan River (1700 South)":           "10171000",
}

OUTFLOW_SITES = {
    "Bear River Canal (diversion)":        "10113500",
    "Ogden-Brigham Canal":                 "10133650",
    "Weber Basin Diversion":               "10141000",   
    "Jordan Narrows (diversion point)":    "10168000",
}

START_DATE = "1950-01-01"
END_DATE   = "2025-12-31"

In [ ]:

def fetch_site(site_id):
    """Pull daily mean discharge (param 00060) for one USGS site."""
    params = {
        "format":      "rdb",        
        "sites":       site_id,
        "startDT":     START_DATE,
        "endDT":       END_DATE,
        "parameterCd": "00060",      # discharge in ft³/s
        "statCd":      "00003",      # daily mean
    }
    response = requests.get(BASE_URL, params=params, timeout=60)
    response.raise_for_status()

    # strip comment lines 
    lines = [l for l in response.text.splitlines() if not l.startswith("#")]
    if len(lines) < 3:
        print(f"  No data for site {site_id}")
        return pd.DataFrame()

    content = "\n".join([lines[0]] + lines[2:])   # header + data rows
    df = pd.read_csv(StringIO(content), sep="\t", low_memory=False)

    # find the discharge column 
    discharge_col = [c for c in df.columns if c.endswith("_00060_00003")]
    if not discharge_col:
        print(f"  No discharge column for site {site_id}")
        return pd.DataFrame()

    df = df[["datetime", discharge_col[0]]].copy()
    df.columns = ["date", "discharge_cfs"]
    df["date"]          = pd.to_datetime(df["date"], errors="coerce")
    df["discharge_cfs"] = pd.to_numeric(df["discharge_cfs"], errors="coerce")
    df = df.dropna(subset=["date"]).sort_values("date").reset_index(drop=True)
    return df

#fetch all sites, print a summary

def fetch_all(site_dict, label):
    print(f"\n── Fetching {label} ──")
    results = {}
    for name, site_id in site_dict.items():
        print(f"  {name} (site {site_id}) ...", end=" ")
        df = fetch_site(site_id)
        if not df.empty:
            print(f"{len(df):,} daily records "
                  f"({df['date'].min().date()} to {df['date'].max().date()})")
        else:
            print("no data")
        results[name] = df
    return results

inflow_raw  = fetch_all(INFLOW_SITES,  "INFLOWS")
outflow_raw = fetch_all(OUTFLOW_SITES, "OUTFLOWS")

In [ ]:
def build_monthly(raw_dict):
    """
    For each site: resample daily → monthly mean.
    Then merge all sites side-by-side on date.
    """
    monthly_frames = []
    for name, df in raw_dict.items():
        if df.empty:
            continue
        monthly = (df.set_index("date")["discharge_cfs"]
                     .resample("MS")        # MS = month-start frequency
                     .mean()
                     .rename(name)
                     .to_frame())
        monthly_frames.append(monthly)

    if not monthly_frames:
        return pd.DataFrame()

    # join all sites on date (outer join to keep all dates, even if some sites are missing)
    combined = monthly_frames[0]
    for frame in monthly_frames[1:]:
        combined = combined.join(frame, how="outer")

    combined.index.name = "date"
    combined = combined.sort_index()
    return combined

inflow_monthly  = build_monthly(inflow_raw)
outflow_monthly = build_monthly(outflow_raw)

# add a total column, save to CSV

# total monthly inflow = sum across all inflow columns (ignores NaN)
inflow_monthly["TOTAL_INFLOW_cfs"]  = inflow_monthly.sum(axis=1)
outflow_monthly["TOTAL_OUTFLOW_cfs"] = outflow_monthly.sum(axis=1)

inflow_path  = os.path.join(OUTPUT_DIR, "gsl_inflow_monthly.csv")
outflow_path = os.path.join(OUTPUT_DIR, "gsl_outflow_monthly.csv")

inflow_monthly.to_csv(inflow_path)
outflow_monthly.to_csv(outflow_path)

print(f"\nSaved inflow  CSV → {inflow_path}")
print(f"Saved outflow CSV → {outflow_path}")

# quick check
print("\n── Inflow monthly (first 5 rows) ──")
print(inflow_monthly.head())

print("\n── Outflow monthly (first 5 rows) ──")
print(outflow_monthly.head())

In [ ]:
OUTPUT_DIR = "/Users/kimlanaghen/Downloads"

df = pd.read_csv(os.path.join(OUTPUT_DIR, "gsl_inflow_monthly.csv"),
                 index_col="date", parse_dates=True)

# get days in each month
days_in_month = df.index.days_in_month

# convert every column from mean cfs → acre-feet for that month
df_acft = df.multiply(days_in_month * 1.9835, axis=0)

# rename columns
df_acft.columns = [c + "_acft" for c in df_acft.columns]

df_acft.to_csv(os.path.join(OUTPUT_DIR, "gsl_inflow_monthly_acft.csv"))
print("Saved acre-feet version")
print(df_acft.head())